In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm
import random
import os

from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing.image import img_to_array,load_img
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.callbacks import EarlyStopping,ModelCheckpoint
from tensorflow.keras.layers import GlobalAveragePooling2D,GlobalMaxPooling2D
from tensorflow.keras.models import Model,Sequential
from tensorflow.keras.optimizers import Adam,SGD,RMSprop
from sklearn.metrics import confusion_matrix,classification_report

In [2]:
!pip install -q kaggle

In [3]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"nabila45","key":"9cb09270561f1217d67c35c440eeedcc"}'}

In [4]:
! mkdir ~/.kaggle/

In [5]:
! cp kaggle.json ~/.kaggle/

In [6]:
! chmod 600 ~/.kaggle/kaggle.json

In [7]:
! kaggle dataset  list

usage: kaggle [-h] [-v] [-W]
              {competitions,c,datasets,d,kernels,k,models,m,files,f,config}
              ...
kaggle: error: argument command: invalid choice: 'dataset' (choose from competitions, c, datasets, d, kernels, k, models, m, files, f, config)


In [8]:
! kaggle datasets download -d kaggle/sf-salaries

Dataset URL: https://www.kaggle.com/datasets/kaggle/sf-salaries
License(s): CC0-1.0
  0% 0.00/11.5M [00:00<?, ?B/s]
100% 11.5M/11.5M [00:00<00:00, 167MB/s]


In [9]:
!unzip sf-salaries.zip

Archive:  sf-salaries.zip
  inflating: Salaries.csv            
  inflating: database.sqlite         


In [10]:
df = pd.read_csv('Salaries.csv')
df.head(5)

/tmp/ipython-input-4249584348.py:1: DtypeWarning: Columns (3,4,5,6,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('Salaries.csv')


,Id,EmployeeName,JobTitle,BasePay,OvertimePay,OtherPay,Benefits,TotalPay,TotalPayBenefits,Year,Notes,Agency,Status
0,1,NATHANIEL FORD,GENERAL MANAGER-METROPOLITAN TRANSIT AUTHORITY,167411.18,0.0,400184.25,NaN,567595.43,567595.43,2011,NaN,San Francisco,NaN
1,2,GARY JIMENEZ,CAPTAIN III (POLICE DEPARTMENT),155966.02,245131.88,137811.38,NaN,538909.28,538909.28,2011,NaN,San Francisco,NaN
2,3,ALBERT PARDINI,CAPTAIN III (POLICE DEPARTMENT),212739.13,106088.18,16452.6,NaN,335279.91,335279.91,2011,NaN,San Francisco,NaN
3,4,CHRISTOPHER CHONG,WIRE ROPE CABLE MAINTENANCE MECHANIC,77916.0,56120.71,198306.9,NaN,332343.61,332343.61,2011,NaN,San Francisco,NaN
4,5,PATRICK GARDNER,"DEPUTY CHIEF OF DEPARTMENT,(FIRE DEPARTMENT)",134401.6,9737.0,182234.59,NaN,326373.19,326373.19,2011,NaN,San Francisco,NaN


In [11]:
df.columns

Index(['Id', 'EmployeeName', 'JobTitle', 'BasePay', 'OvertimePay', 'OtherPay',
       'Benefits', 'TotalPay', 'TotalPayBenefits', 'Year', 'Notes', 'Agency',
       'Status'],
      dtype='object')

In [12]:
df.shape

(148654, 13)

In [13]:
df.isnull().sum()

,0
Id,0
EmployeeName,0
JobTitle,0
BasePay,605
OvertimePay,0
OtherPay,0
Benefits,36159
TotalPay,0
TotalPayBenefits,0
Year,0


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148654 entries, 0 to 148653
Data columns (total 13 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Id                148654 non-null  int64  
 1   EmployeeName      148654 non-null  object 
 2   JobTitle          148654 non-null  object 
 3   BasePay           148049 non-null  object 
 4   OvertimePay       148654 non-null  object 
 5   OtherPay          148654 non-null  object 
 6   Benefits          112495 non-null  object 
 7   TotalPay          148654 non-null  float64
 8   TotalPayBenefits  148654 non-null  float64
 9   Year              148654 non-null  int64  
 10  Notes             0 non-null       float64
 11  Agency            148654 non-null  object 
 12  Status            38119 non-null   object 
dtypes: float64(3), int64(2), object(8)
memory usage: 14.7+ MB


In [15]:
df['EmployeeName'].value_counts()

,count
EmployeeName,
Kevin Lee,13
William Wong,11
Richard Lee,11
Steven Lee,11
John Chan,9
...,...
Saoirse C Brownfield,1
Miguel P Lucana,1
Johnmark L Henderson,1


In [16]:
df['JobTitle'].value_counts()

,count
JobTitle,
Transit Operator,7036
Special Nurse,4389
Registered Nurse,3736
Public Svc Aide-Public Works,2518
Police Officer 3,2421
...,...
Light Rail Vehicle Equip Eng,1
Civil Case Settlmnt Specialist,1
"ADMINISTRATOR, SFGH MEDICAL CENTER",1


In [17]:
df['OvertimePay'].value_counts()

,count
OvertimePay,
0.0,66103
0.00,11218
681.23,41
10.68,41
152.13,38
...,...
493.47,1
4842.72,1
29661.41,1


Total number of job titles contain captain

In [18]:
len(df[df['JobTitle'].str.contains('CAPTAIN')])

141

In [20]:
df[df['JobTitle'].str.contains('CAPTAIN')].count()

,0
Id,141
EmployeeName,141
JobTitle,141
BasePay,141
OvertimePay,141
OtherPay,141
Benefits,0
TotalPay,141
TotalPayBenefits,141
Year,141


ALL the employee name from fire department

In [24]:
len(df[df['JobTitle'].str.contains('FIRE DEPARTMENT')])

222

In [27]:
df[df['JobTitle'].str.contains('FIRE DEPARTMENT')]['EmployeeName']

,EmployeeName
4,PATRICK GARDNER
6,ALSON LEE
8,MICHAEL MORRIS
9,JOANNE HAYES-WHITE
10,ARTHUR KENNEY
...,...
4955,AUDRY LEE
5498,VINCENT PEREZ
8436,JENSEN RHODES
16285,AARON STEVENSON


Minimum maximum avrage basepay

In [28]:
df.columns

Index(['Id', 'EmployeeName', 'JobTitle', 'BasePay', 'OvertimePay', 'OtherPay',
       'Benefits', 'TotalPay', 'TotalPayBenefits', 'Year', 'Notes', 'Agency',
       'Status'],
      dtype='object')

In [29]:
df['BasePay'].describe()

,BasePay
count,148049.0
unique,109900.0
top,0.0
freq,875.0


Replace not provided in employee name column to NAN

In [32]:
import numpy as np
df['EmployeeName'].replace('Not provided',np.nan)

,EmployeeName
0,NATHANIEL FORD
1,GARY JIMENEZ
2,ALBERT PARDINI
3,CHRISTOPHER CHONG
4,PATRICK GARDNER
...,...
148649,Roy I Tillery
148650,NaN
148651,NaN
148652,NaN


Drop the rows having missing values

In [41]:
df[df.isnull().sum(axis=1)==3]

,Id,EmployeeName,JobTitle,BasePay,OvertimePay,OtherPay,Benefits,TotalPay,TotalPayBenefits,Year,Notes,Agency,Status
0,1,NATHANIEL FORD,GENERAL MANAGER-METROPOLITAN TRANSIT AUTHORITY,167411.18,0.0,400184.25,NaN,567595.43,567595.43,2011,NaN,San Francisco,NaN
1,2,GARY JIMENEZ,CAPTAIN III (POLICE DEPARTMENT),155966.02,245131.88,137811.38,NaN,538909.28,538909.28,2011,NaN,San Francisco,NaN
2,3,ALBERT PARDINI,CAPTAIN III (POLICE DEPARTMENT),212739.13,106088.18,16452.6,NaN,335279.91,335279.91,2011,NaN,San Francisco,NaN
3,4,CHRISTOPHER CHONG,WIRE ROPE CABLE MAINTENANCE MECHANIC,77916.0,56120.71,198306.9,NaN,332343.61,332343.61,2011,NaN,San Francisco,NaN
4,5,PATRICK GARDNER,"DEPUTY CHIEF OF DEPARTMENT,(FIRE DEPARTMENT)",134401.6,9737.0,182234.59,NaN,326373.19,326373.19,2011,NaN,San Francisco,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
110526,110527,Arthur L Curry,PS Aide Health Services,NaN,0.0,10.67,0.0,10.67,10.67,2013,NaN,San Francisco,NaN
110527,110528,Nereida Vega,Senior Clerk,NaN,0.0,5.56,0.0,5.56,5.56,2013,NaN,San Francisco,NaN
110528,110529,Timothy E Gibson,Police Officer 3,NaN,0.0,0.0,-2.73,0.00,-2.73,2013,NaN,San Francisco,NaN
110529,110530,Mark E Laherty,Police Officer 3,NaN,0.0,0.0,-8.2,0.00,-8.20,2013,NaN,San Francisco,NaN


In [45]:
df.drop(index=df[df.isnull().sum(axis=1)==3].index, inplace=True)

find the job title of albert pardini

In [50]:
df[df['EmployeeName']=='Nereida Vega']['JobTitle']

,JobTitle
72353,Senior Clerk


display the person having highest basepay

In [53]:
df['BasePay'] = pd.to_numeric(df['BasePay'], errors='coerce')
idx_max_basepay = df['BasePay'].idxmax()
employee_with_highest_basepay = df.loc[idx_max_basepay, 'EmployeeName']

print(f"The employee with the highest BasePay is: {employee_with_highest_basepay}")
print(f"Their BasePay is: {df.loc[idx_max_basepay, 'BasePay']}")

The employee with the highest BasePay is: Gregory P Suhr
Their BasePay is: 319275.01


In [56]:
df[df['BasePay'].max()==df['BasePay']] ['EmployeeName']

,EmployeeName
72925,Gregory P Suhr


find avrage basepay of all employee per year

In [58]:
df.groupby('Year')['BasePay'].mean()

,BasePay
Year,
2012,65436.406857
2013,69630.030216
2014,66564.421924
